### Notebook 3a — Synthetic Image Generation with DC-GAN
---
>Train two DC-GANs (one for Pneumonia, one for No Finding) **only on the real 5K training subset** defined in Notebook 2, and generate the synthetic images used in scenarios 2 and 3: 5,000 Pneumonia + 2,500 No Finding. The notebook also runs quick screening checks (visual inspection, intensity statistics and a nearest-neighbour memorisation test). Formal quality metrics (FID/KID, real-vs-synthetic detector) are computed in Notebook 3c.

### 1. Configuration and Loading

**Design decisions**

* **Training data = `real_5k.csv`** (683 positives + 4,317 matched negatives). It is exactly the real data the baseline classifier sees, so the synthetic images contain no real information beyond the baseline. Validation and test patients are never used to train the generators.
* **Two unconditional DC-GANs**, one per class, so the class of every synthetic image is known by construction.
* **Resolution 128×128.** A DC-GAN at 1024×1024 is not realistic with 683 images; the same downsampling pipeline is applied to the real images used for training and for the checks.
* **DiffAugment + EMA** (differentiable augmentation on the discriminator inputs and an exponential moving average of the generator). They do not change the DC-GAN architecture; they are the standard way to avoid discriminator over-fitting with fewer than 1,000 images. No horizontal flips are used (they would swap heart/stomach sides).

Attach as input (1) the splits exported by Notebook 2 (a private Dataset or the Notebook 2 output) and (2) the NIH ChestX-ray14 dataset (the CSV `path` column points to it). `SPLITS_DIR` is detected automatically by searching for `real_5k.csv`.

In [ ]:
import os, time, math, json, random
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
import torch, torch.nn as nn, torch.nn.functional as F
from torchvision.utils import make_grid

SEED = 42
# --- Locate the splits exported by Notebook 2 (works for a private Dataset or for a Notebook-output input) ---
from glob import glob
_hits = [h for depth in range(1, 5) for h in glob('/kaggle/input/' + '*/' * depth + 'real_5k.csv')]
assert _hits, 'real_5k.csv not found under /kaggle/input: attach the splits (Dataset or Notebook 2 output) as input'
SPLITS_DIR = os.path.dirname(_hits[0])
print('SPLITS_DIR =', SPLITS_DIR)
WORK = Path('/kaggle/working')
OUT_DIR, CKPT_DIR, CACHE_DIR = WORK / 'synthetic' / 'dcgan', WORK / 'checkpoints' / 'dcgan', WORK / 'cache'

IMG_SIZE = 128                       # 64 / 128 / 256 (must be a power of two)
NZ, NGF, NDF = 100, 64, 64
BATCH = 64
ITERS = {'pos': 30_000, 'neg': 20_000}
LR, BETAS, EMA_DECAY, REAL_LABEL = 2e-4, (0.5, 0.999), 0.999, 0.9
N_SYN = {'pos': 5000, 'neg': 2500}   # synthetic images to generate
CKPT_EVERY, LOG_EVERY, RESUME = 5000, 200, True

for d in (OUT_DIR, CKPT_DIR, CACHE_DIR):
    d.mkdir(parents=True, exist_ok=True)

def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)
set_seed(SEED)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
torch.backends.cudnn.benchmark = True
print('Device:', DEVICE, '|', torch.cuda.get_device_name(0) if DEVICE.type == 'cuda' else 'CPU (training will be very slow)')

# --- Manifests from Notebook 2 ---
real_5k = pd.read_csv(f'{SPLITS_DIR}/real_5k.csv')
val_full = pd.read_csv(f'{SPLITS_DIR}/real_val_full.csv')
test_full = pd.read_csv(f'{SPLITS_DIR}/real_test_full.csv', usecols=['image', 'Patient ID'])

# --- Leakage and consistency checks ---
assert not (set(real_5k['Patient ID']) & set(val_full['Patient ID'])), 'patient leakage 5K/val'
assert not (set(real_5k['Patient ID']) & set(test_full['Patient ID'])), 'patient leakage 5K/test'
assert (real_5k.loc[real_5k.label == 0, 'Finding Labels'] == 'No Finding').all(), 'training negatives must be No Finding'
assert os.path.exists(real_5k['path'].iloc[0]), 'Image path not found: attach the NIH ChestX-ray14 dataset as input'

print(f"Generator training data: {int((real_5k.label == 1).sum())} positives | {int((real_5k.label == 0).sum())} negatives")

### 2. Real Image Cache

Images are converted to 8-bit grayscale and resized (Lanczos, anti-aliased) once, then cached as `.npz` so the generators train from memory. The validation references (all positives and a random sample of 500 clean negatives) are used only for the memorisation check in section 8, never for training.

In [ ]:
def load_gray(path, size):
    im = Image.open(path).convert('L')
    return np.asarray(im.resize((size, size), Image.Resampling.LANCZOS), dtype=np.uint8)

def load_many(paths, size, workers=8):
    with ThreadPoolExecutor(workers) as ex:
        return np.stack(list(ex.map(lambda p: load_gray(p, size), paths)))

cache_file = CACHE_DIR / f'real_{IMG_SIZE}.npz'
if cache_file.exists():
    z = np.load(cache_file); REAL = {k: z[k] for k in z.files}
    print('Loaded cache:', cache_file)
else:
    t0 = time.time()
    val_pos = val_full[val_full.label == 1]
    val_neg = val_full[(val_full.label == 0) & (~val_full.hard_negative)].sample(500, random_state=SEED)
    REAL = {'pos': load_many(real_5k.loc[real_5k.label == 1, 'path'], IMG_SIZE),
            'neg': load_many(real_5k.loc[real_5k.label == 0, 'path'], IMG_SIZE),
            'val_pos': load_many(val_pos['path'], IMG_SIZE),
            'val_neg': load_many(val_neg['path'], IMG_SIZE)}
    np.savez_compressed(cache_file, **REAL)
    print(f'Cached in {time.time() - t0:.0f}s ->', cache_file)

print({k: v.shape for k, v in REAL.items()})

### 3. Model Definitions

Standard DC-GAN (Radford et al., 2016): transposed-convolution generator with BatchNorm + ReLU and a strided-convolution discriminator with BatchNorm + LeakyReLU, adapted to any power-of-two resolution. The loss is the non-saturating GAN loss with one-sided label smoothing (0.9).

`diffaug` applies brightness, contrast, translation and cut-out **differentiably** to every image the discriminator sees (real and fake), so the discriminator cannot memorise the 683 real images.

In [ ]:
def conv_init(m):
    if isinstance(m, (nn.Conv2d, nn.ConvTranspose2d)):
        nn.init.normal_(m.weight, 0.0, 0.02)
    elif isinstance(m, nn.BatchNorm2d):
        nn.init.normal_(m.weight, 1.0, 0.02); nn.init.zeros_(m.bias)

class Generator(nn.Module):
    def __init__(self, nz=NZ, ngf=NGF, nc=1, size=IMG_SIZE):
        super().__init__()
        depth = int(math.log2(size)) - 2                      # doublings from 4x4 to size
        ch = ngf * 2 ** (depth - 1)
        layers = [nn.ConvTranspose2d(nz, ch, 4, 1, 0, bias=False), nn.BatchNorm2d(ch), nn.ReLU(True)]
        for _ in range(depth - 1):
            layers += [nn.ConvTranspose2d(ch, ch // 2, 4, 2, 1, bias=False), nn.BatchNorm2d(ch // 2), nn.ReLU(True)]
            ch //= 2
        layers += [nn.ConvTranspose2d(ch, nc, 4, 2, 1, bias=False), nn.Tanh()]
        self.net = nn.Sequential(*layers); self.apply(conv_init)
    def forward(self, z): return self.net(z)

class Discriminator(nn.Module):
    def __init__(self, ndf=NDF, nc=1, size=IMG_SIZE):
        super().__init__()
        depth = int(math.log2(size)) - 2
        layers, ch = [nn.Conv2d(nc, ndf, 4, 2, 1, bias=False), nn.LeakyReLU(0.2, True)], ndf
        for _ in range(depth - 1):
            layers += [nn.Conv2d(ch, ch * 2, 4, 2, 1, bias=False), nn.BatchNorm2d(ch * 2), nn.LeakyReLU(0.2, True)]
            ch *= 2
        layers += [nn.Conv2d(ch, 1, 4, 1, 0, bias=False)]
        self.net = nn.Sequential(*layers); self.apply(conv_init)
    def forward(self, x): return self.net(x).view(-1)

def diffaug(x, translate=0.125, cutout=0.35):
    # Differentiable augmentation (Zhao et al., 2020): brightness, contrast, translation, cut-out
    B, _, S, _ = x.shape; dev = x.device
    x = x + (torch.rand(B, 1, 1, 1, device=dev) - 0.5) * 0.4
    m = x.mean(dim=(1, 2, 3), keepdim=True)
    x = (x - m) * (torch.rand(B, 1, 1, 1, device=dev) * 0.5 + 0.75) + m
    sh = int(S * translate + 0.5)
    tx, ty = torch.randint(-sh, sh + 1, (B, 1, 1), device=dev), torch.randint(-sh, sh + 1, (B, 1, 1), device=dev)
    gb, gx, gy = torch.meshgrid(torch.arange(B, device=dev), torch.arange(S, device=dev), torch.arange(S, device=dev), indexing='ij')
    gx, gy = torch.clamp(gx + tx + 1, 0, S + 1), torch.clamp(gy + ty + 1, 0, S + 1)
    x = F.pad(x, [1, 1, 1, 1]).permute(0, 2, 3, 1).contiguous()[gb, gx, gy].permute(0, 3, 1, 2)
    cs = int(S * cutout + 0.5)
    ox, oy = torch.randint(0, S + (1 - cs % 2), (B, 1, 1), device=dev), torch.randint(0, S + (1 - cs % 2), (B, 1, 1), device=dev)
    gb, gx, gy = torch.meshgrid(torch.arange(B, device=dev), torch.arange(cs, device=dev), torch.arange(cs, device=dev), indexing='ij')
    gx, gy = torch.clamp(gx + ox - cs // 2, 0, S - 1), torch.clamp(gy + oy - cs // 2, 0, S - 1)
    mask = torch.ones(B, S, S, device=dev); mask[gb, gx, gy] = 0
    return x * mask.unsqueeze(1)

@torch.no_grad()
def ema_update(ema, model, decay):
    for pe, p in zip(ema.parameters(), model.parameters()): pe.lerp_(p, 1 - decay)
    for be, b in zip(ema.buffers(), model.buffers()): be.copy_(b)

print('G params: %.1fM | D params: %.1fM' % (sum(p.numel() for p in Generator().parameters()) / 1e6,
                                             sum(p.numel() for p in Discriminator().parameters()) / 1e6))

### 4. Training Loop

Batches are sampled at random from the in-memory tensor (iteration-based training, because 683 images are only ~11 batches per epoch). Every `CKPT_EVERY` iterations the loop saves a checkpoint (generator, EMA generator, discriminator and optimisers) and a grid generated from a **fixed** noise vector, so the evolution of the generator can be compared visually. If the Kaggle session is interrupted, running the cell again resumes from the last checkpoint (`RESUME = True`).

In [ ]:
bce = nn.BCEWithLogitsLoss()

def to_tensor(u8):                                   # uint8 (N,H,W) -> float [-1,1] (N,1,H,W)
    return torch.from_numpy(u8).float().div(127.5).sub(1).unsqueeze(1)

def save_grid(x, path, nrow=8):
    g = make_grid(x.cpu(), nrow=nrow, normalize=True, value_range=(-1, 1), padding=2)
    Image.fromarray((g[0].numpy() * 255).astype(np.uint8)).save(path)

def train_dcgan(name, real_u8, iters):
    set_seed(SEED)
    data = to_tensor(real_u8).to(DEVICE); N = len(data)
    G, D = Generator().to(DEVICE), Discriminator().to(DEVICE)
    G_ema = Generator().to(DEVICE); G_ema.load_state_dict(G.state_dict()); G_ema.eval()
    oG = torch.optim.Adam(G.parameters(), LR, betas=BETAS); oD = torch.optim.Adam(D.parameters(), LR, betas=BETAS)
    fixed_z = torch.randn(64, NZ, 1, 1, generator=torch.Generator().manual_seed(SEED)).to(DEVICE)
    hist, start = [], 0
    last = CKPT_DIR / f'{name}_last.pt'
    if RESUME and last.exists():
        ck = torch.load(last, map_location=DEVICE)
        G.load_state_dict(ck['G']); D.load_state_dict(ck['D']); G_ema.load_state_dict(ck['G_ema'])
        oG.load_state_dict(ck['oG']); oD.load_state_dict(ck['oD']); hist, start = ck['hist'], ck['it']
        print(f'[{name}] resumed from iteration {start}')
    (OUT_DIR / 'samples').mkdir(exist_ok=True)
    t0 = time.time()
    for it in range(start, iters):
        real = data[torch.randint(0, N, (BATCH,), device=DEVICE)]
        z = torch.randn(BATCH, NZ, 1, 1, device=DEVICE)
        fake = G(z)
        # --- Discriminator ---
        d_real, d_fake = D(diffaug(real)), D(diffaug(fake.detach()))
        lD = bce(d_real, torch.full_like(d_real, REAL_LABEL)) + bce(d_fake, torch.zeros_like(d_fake))
        oD.zero_grad(set_to_none=True); lD.backward(); oD.step()
        # --- Generator (non-saturating loss) ---
        g_out = D(diffaug(fake))
        lG = bce(g_out, torch.ones_like(g_out))
        oG.zero_grad(set_to_none=True); lG.backward(); oG.step()
        ema_update(G_ema, G, min(EMA_DECAY, (1 + it) / (10 + it)))
        if (it + 1) % LOG_EVERY == 0:
            hist.append((it + 1, lD.item(), lG.item(), torch.sigmoid(d_real).mean().item(), torch.sigmoid(d_fake).mean().item()))
        if (it + 1) % CKPT_EVERY == 0 or it + 1 == iters:
            with torch.no_grad(): save_grid(G_ema(fixed_z), OUT_DIR / 'samples' / f'{name}_it{it + 1:06d}.png')
            ck = {'G': G.state_dict(), 'D': D.state_dict(), 'G_ema': G_ema.state_dict(), 'oG': oG.state_dict(),
                  'oD': oD.state_dict(), 'hist': hist, 'it': it + 1}
            torch.save(ck, last); torch.save(G_ema.state_dict(), CKPT_DIR / f'{name}_ema_it{it + 1:06d}.pt')
            print(f'[{name}] it {it + 1:6d}/{iters} | lD {lD.item():.3f} lG {lG.item():.3f} | '
                  f'D(real) {hist[-1][3]:.2f} D(fake) {hist[-1][4]:.2f} | {(time.time() - t0) / 60:.1f} min')
    torch.save(G_ema.state_dict(), CKPT_DIR / f'{name}_ema_final.pt')
    return pd.DataFrame(hist, columns=['it', 'loss_D', 'loss_G', 'D_real', 'D_fake'])

### 5. Train the Pneumonia DC-GAN

Trained on the 683 real positives. Monitor `D(real)` and `D(fake)`: healthy training keeps them away from 1.0 / 0.0 respectively; `D(real) → 1` and `D(fake) → 0` means the discriminator is winning and the generator has collapsed or stopped learning.

In [ ]:
hist_pos = train_dcgan('pos', REAL['pos'], ITERS['pos'])

### 6. Train the No-Finding DC-GAN

Trained on the 4,317 real negatives from `real_5k`, which are matched to the positives on view position, sex, age and follow-up. The synthetic negatives therefore inherit that matched profile.

In [ ]:
hist_neg = train_dcgan('neg', REAL['neg'], ITERS['neg'])

### 7. Training Curves and Visual Inspection

Left: losses and discriminator outputs. Right: real images next to samples from the final EMA generator (same noise-free comparison, no cherry-picking: the first 32 images of each).

In [ ]:
@torch.no_grad()
def load_ema(name, ckpt=None):
    G = Generator().to(DEVICE); G.load_state_dict(torch.load(CKPT_DIR / (ckpt or f'{name}_ema_final.pt'), map_location=DEVICE)); return G.eval()

@torch.no_grad()
def sample(G, n, seed, bs=250):
    g = torch.Generator(device=DEVICE).manual_seed(seed); out = []
    for i in range(0, n, bs):
        z = torch.randn(min(bs, n - i), NZ, 1, 1, device=DEVICE, generator=g)
        out.append(((G(z).clamp(-1, 1) + 1) * 127.5).round().byte().squeeze(1).cpu())
    return torch.cat(out).numpy()

def u8_grid(arr, n=32, nrow=8):
    x = torch.from_numpy(arr[:n]).float().unsqueeze(1) / 255
    return make_grid(x, nrow=nrow, padding=2)[0].numpy()

fig, ax = plt.subplots(2, 3, figsize=(17, 10))
for r, (name, h) in enumerate([('pos', hist_pos), ('neg', hist_neg)]):
    ax[r, 0].plot(h.it, h.loss_D, label='loss D'); ax[r, 0].plot(h.it, h.loss_G, label='loss G')
    ax[r, 0].plot(h.it, h.D_real, '--', label='D(real)'); ax[r, 0].plot(h.it, h.D_fake, '--', label='D(fake)')
    ax[r, 0].set_title(f'{name}: training curves'); ax[r, 0].legend()
    ax[r, 1].imshow(u8_grid(REAL[name]), cmap='gray'); ax[r, 1].set_title(f'{name}: REAL'); ax[r, 1].axis('off')
    ax[r, 2].imshow(u8_grid(sample(load_ema(name), 32, SEED)), cmap='gray'); ax[r, 2].set_title(f'{name}: SYNTHETIC (EMA)'); ax[r, 2].axis('off')
plt.tight_layout(); plt.show()

### 8. Generate the Synthetic Dataset

Generates **5,000 Pneumonia + 2,500 No Finding** images with the final EMA generators, with a fixed seed per class for reproducibility. Images are saved as 8-bit PNG (plus a `.npy` array per class for fast loading in later notebooks) and described in `dcgan_manifest.csv`. For scenarios 2 and 4 the first 2,500 positives and all 2,500 negatives are used (`subset` column); for scenarios 3 and 5 all 5,000 positives are used.

In [ ]:
SYN = {}
rows = []
for cls, label, seed in [('pos', 1, SEED), ('neg', 0, SEED + 1)]:
    G = load_ema(cls); arr = sample(G, N_SYN[cls], seed); SYN[cls] = arr
    d = OUT_DIR / ('pneumonia' if label else 'no_finding'); d.mkdir(exist_ok=True)
    np.save(OUT_DIR / f'dcgan_{cls}_{IMG_SIZE}.npy', arr)
    for i, a in enumerate(arr):
        fn = f'dcgan_{cls}_{i:05d}.png'
        Image.fromarray(a, 'L').save(d / fn)
        rows.append({'image': fn, 'path': str(d / fn), 'label': label, 'source': 'synthetic', 'generator': 'dcgan',
                     'img_size': IMG_SIZE, 'gen_seed': seed, 'index': i,
                     'subset': 'only_2500' if (label == 0 or i < 2500) else 'extra_pos'})
manifest = pd.DataFrame(rows)
manifest.to_csv(OUT_DIR / 'dcgan_manifest.csv', index=False)
print(manifest.groupby(['label', 'subset']).size().rename('images'))

### 9. Screening Checks

**Intensity statistics.** Large differences in mean/std between real and synthetic images are a trivial shortcut for the classifier (source = class).

**Nearest-neighbour memorisation test.** For each synthetic image, distance (on 32×32 downsampled pixels) to its nearest *training* image, compared with the same distance for real held-out validation images. If synthetic images are much closer to the training set than real unseen images are, the generator is probably copying. The pairs with the smallest distance are shown below. This is a rough pixel-space screen (smooth images are also closer to everything), so the formal test with deep features is done in Notebook 3c.

In [ ]:
stats = pd.DataFrame({
    'real_train':   {c: (REAL[c].mean(), REAL[c].std()) for c in ['pos', 'neg']},
    'synthetic':    {c: (SYN[c].mean(), SYN[c].std()) for c in ['pos', 'neg']}}).T
print('Pixel intensity (mean, std) per class:'); print(stats.map(lambda t: tuple(round(v, 1) for v in t)))

def small(u8, s=32):
    return F.adaptive_avg_pool2d(torch.from_numpy(u8).float().unsqueeze(1) / 255, s).flatten(1)

def nn_dist(q, ref, chunk=1000):
    return torch.cat([torch.cdist(q[i:i + chunk], ref).min(1).values for i in range(0, len(q), chunk)])

fig, ax = plt.subplots(2, 8, figsize=(20, 5.5)); res = []
for r, (cls, vref) in enumerate([('pos', 'val_pos'), ('neg', 'val_neg')]):
    tr, syn, val = small(REAL[cls]), small(SYN[cls]), small(REAL[vref])
    d_syn, d_val = nn_dist(syn, tr), nn_dist(val, tr)
    res.append({'class': cls, 'syn->train median': d_syn.median().item(), 'val->train median': d_val.median().item(),
                'syn->train p1': d_syn.quantile(0.01).item(), 'val->train p1': d_val.quantile(0.01).item(),
                '% syn closer than val p1': 100 * (d_syn < d_val.quantile(0.01)).float().mean().item()})
    idx = d_syn.argsort()[:4]; nn_idx = torch.cdist(syn[idx], tr).argmin(1)
    for k in range(4):
        ax[r, 2 * k].imshow(SYN[cls][idx[k]], cmap='gray'); ax[r, 2 * k].set_title(f'{cls} synthetic (d={d_syn[idx[k]]:.2f})', fontsize=9)
        ax[r, 2 * k + 1].imshow(REAL[cls][nn_idx[k]], cmap='gray'); ax[r, 2 * k + 1].set_title('nearest real train', fontsize=9)
for a in ax.ravel(): a.axis('off')
plt.tight_layout(); plt.show()
print(pd.DataFrame(res).round(3).to_string(index=False))

### 10. Export

Everything needed downstream is in `/kaggle/working/synthetic/dcgan/` (PNGs, `.npy` arrays, `dcgan_manifest.csv`, training samples) and `/kaggle/working/checkpoints/dcgan/` (EMA checkpoints, one every `CKPT_EVERY` iterations). To use them in Notebooks 3c and 4+, **commit the notebook ("Save Version") and publish the output as a private Kaggle Dataset** (e.g. `synthetic-dcgan`), as done with the splits.

In [ ]:
total = sum(f.stat().st_size for f in OUT_DIR.rglob('*') if f.is_file()) / 1e6
print(f'Synthetic output: {len(manifest):,} images, {total:.0f} MB in {OUT_DIR}')
print('Checkpoints:', sorted(p.name for p in CKPT_DIR.glob('*_ema_*.pt'))[:4], '...')